# Channel-Attention 3D CNN and Random Forest Soft Voting for Cell-Candidate Filtering and Lineage Tracking

# A Hybrid Deep Learning and Classical Pipeline for Cell Tracking During Development: A Research Narrative

# Abstract

We present a complete, single-cell-runnable pipeline for automated cell tracking in three-dimensional developmental imaging. The system integrates unsupervised candidate region proposal, a 3D convolutional neural network (CNN) with channel attention, a classical RandomForest classifier, soft-voting ensemble filtering, Hungarian frame-to-frame tracking with gated division detection, and an exact re-implementation of the competition’s combined tracking metric. On the candidate classification task, the best model achieved a test accuracy of 95.7%, an ROC AUC of 0.997, and a confusion matrix with 183 true negatives, 9 false positives, 0 false negatives, and 18 true positives. Accuracy across data splits was 98.0% (train), 100% (validation), 95.7% (test), and 99.8% (holdout/unseen). Generalization gaps were small, and training loss decreased smoothly from approximately 0.56 to 0.16 over 15 epochs. These results indicate that the hybrid detector/classifier is highly effective at filtering cell candidates, with excellent generalization to unseen embryos. The final competition tracking score, however, depends on the full tracking and division metric, which is implemented but reported only as a bounded proxy in this run.

# 1. Introduction
   
Tracking cells during embryonic development is a fundamental problem in developmental biology and computational bioimaging. The task requires detecting individual cells across time, linking them into lineages, and identifying division events, all while handling anisotropic voxel spacing, dense packing, noise, and sparse ground-truth annotations. The Biohub – Cell Tracking During Development competition formalizes this challenge by providing 3D+time volumes in Zarr v3 format and ground-truth lineage graphs in GEFF format. The scoring metric combines an adjusted Edge Jaccard term with a weighted Division Jaccard term, making both detection and lineage reconstruction critical.
We developed a complete pipeline that avoids external dependencies beyond the provided Kaggle Python environment.

It uses tensorstore for Zarr v3 reading, pure-standard-library JSON for metadata, and a combination of classical image processing and deep learning for candidate detection and filtering. This narrative focuses on the algorithmic design and the empirical results shown in the provided figures, especially the candidate classifier’s training dynamics, confusion matrix, ROC curve, generalization gap, and cross-split accuracy.

# 2.  Materials and Methods

# 2.1 Data Discovery and Reading

The pipeline recursively discovers .zarr volumes and .geff ground-truth graphs under the competition root, separating train, test, and unknown splits. Volumes are read lazily with tensorstore, using the zarr3 driver with a fallback to the legacy zarr driver. Ground-truth nodes are loaded as (t, z, y, x) coordinates, and the estimated_number_of_nodes metadata is extracted when available; otherwise, the annotated node count is used as a documented lower-bound fallback.

# 2.2 Candidate Region Proposal

For each timepoint, the 3D volume undergoes percentile normalization, anisotropic background subtraction, Difference-of-Gaussians blob enhancement, Otsu thresholding, and anisotropic distance-transform watershed splitting. This produces candidate regions with sub-voxel intensity-weighted centroids and handcrafted features: area, mean intensity, max intensity, and extent. The heavy filtering steps are run on a spatially downsampled copy (1, 2, 2) for speed, while centroids and patches are mapped back to full resolution. A genuine area-threshold fallback is used if no trained model is available.

# 2.3 Classifier Architecture and Ensemble

The deep model is a small 3D CNN with channel attention (squeeze-and-excitation style), batch normalization, max pooling, dropout, and Gaussian input noise during training. It takes (9, 25, 25) voxel patches and outputs a single logit. A classical RandomForest classifier is trained on the four handcrafted features. When both are available, they are combined by soft voting; otherwise, the available model or the area-threshold fallback is used.

# 2.4 Training Protocol

Training uses a leakage-free four-way split at the dataset (embryo/sample) level: 40% train, 15% validation, 15% test, and 30% holdout. The holdout is strictly withheld from training and hyperparameter selection. Outlier detection with IsolationForest downweights anomalous candidates. Randomized hyperparameter search explores learning rate, dropout, and weight decay. The final CNN is trained with early stopping, ReduceLROnPlateau, gradient clipping, label smoothing, and best-checkpoint saving. Model selection compares CNN-only, RF-only, and ensemble by validation accuracy.

# 2.5 Tracking and Metric

Frame-to-frame tracking uses Hungarian assignment with a gated maximum link distance. A second pass re-examines unmatched new detections against frontier nodes, enabling gated division detection when a parent acquires two outgoing edges. The competition metric is re-implemented exactly: per-timepoint bipartite node matching, adjusted Edge Jaccard with 

# 3. Results
   
# 3.1 Training Dynamics

Figure 4 shows the training loss over 15 epochs. The loss decreases smoothly from approximately 0.56 to 0.16, with no divergence or plateau, indicating stable optimization. The curve is consistent with effective regularization: dropout, weight decay, label smoothing, and early stopping all contribute to preventing overfitting. Figure 1, which reports training/validation accuracy, is consistent with this behavior, although the exact curves are not reproduced in the provided images.

# 3.2 Classification Performance

Figure 2 presents the confusion matrix on the test split. The model achieved:
            •	True negatives: 183
            •	False positives: 9
            •	False negatives: 0
            •	True positives: 18

This corresponds to an accuracy of 95.7%, precision of 66.7%, recall of 100%, specificity of 95.3%, and F1 score of 0.80. The zero false-negative rate is particularly important for tracking: missing a true cell candidate can break a lineage, whereas a small number of false positives can often be filtered or tolerated by the tracking and scoring stages. The nine false positives suggest that the ensemble is conservative in the sense of not missing positives, but it occasionally accepts non-cell regions.
Figure 5 shows the ROC curve for the test set, with an AUC of 0.997. This near-perfect ranking performance indicates that the model separates positive and negative candidates extremely well across thresholds. The high AUC also suggests that the default threshold of 0.5 is not the only viable operating point; if higher precision is desired, the threshold can be raised with minimal loss of recall.

# 3.3 Generalization Across Splits

Figure 6 reports accuracy across data splits: train 0.980, validation 1.000, test 0.957, and holdout 0.998. The holdout split is especially informative because it was strictly unseen during training and hyperparameter selection. Its accuracy of 99.8% is nearly identical to the validation accuracy, indicating that the model generalizes well to new embryos. The test split accuracy of 95.7% is slightly lower, which is consistent with the confusion matrix and may reflect sampling variance or a slightly harder test subset.
Figure 3 shows the generalization gap. The train–validation gap and train–holdout gap is both very small, on the order of 0.02 or less. A small gap indicates that the model is not memorizing the training set and that the regularization strategy is effective. The near-zero train–holdout gap is particularly encouraging because it suggests that performance on completely unseen biological samples is stable.

# 4. Discussion
   
The results demonstrate that the hybrid candidate-filtering approach is highly effective. The CNN with channel attention learns discriminative 3D morphology, while the RandomForest provides a complementary classical view based on area, intensity, and extent. Soft voting combines their strengths. The four-way split at the dataset level prevents leakage from correlated patches of the same embryo, which is a common pitfall in cell tracking pipelines. The holdout accuracy of 99.8% is strong evidence that the model does not overfit to specific embryos.

The zero false-negative rate on the test split is a desirable property for a tracking pipeline, because missed detections directly reduce edge recall and can fragment lineages. The nine false positives, while not ideal, are less harmful in this context: they may create spurious nodes or edges, but the tracking metric’s node-count penalty and edge Jaccard are designed to handle some level of over-detection. If precision becomes limiting, the ensemble threshold could be tuned upward, or additional hard-negative mining could be used.
Several limitations should be noted. First, the reported accuracies are for candidate classification, not for the final competition tracking metric. 

The full metric depends on frame-to-frame linking, division detection, and the adjusted Edge Jaccard, which are implemented but not fully reported here. Second, the pipeline operates under a hard runtime budget of three hours and subsamples up to 30 annotated datasets for classifier training, which may limit diversity. Third, the ground-truth annotations are sparse; the labeling strategy deliberately skips ambiguous unmatched regions to avoid manufacturing false negatives, which is methodologically sound but means the negative class may be under-represented in difficult cases. Fourth, the tracking validation is a bounded proxy with capped datasets and frames, so it is not the exact full-video score.

Despite these limitations, the classifier results are promising. The high AUC, high recall, and strong holdout accuracy suggest that the detection and filtering stages are reliable. The exact metric re-implementation and submission writer ensure that the pipeline can produce competition-compliant outputs. Future work should evaluate the full tracking metric on the holdout split, tune the ensemble threshold for the combined score rather than classification accuracy alone, and potentially increase the number of training datasets if runtime permits.

5. Conclusion
   
We have described a complete cell tracking pipeline for developmental imaging, combining classical blob detection, a 3D channel-attention CNN, a RandomForest, and Hungarian tracking with gated division detection. The candidate classifier achieved 95.7% test accuracy, 0.997 ROC AUC, and 99.8% holdout accuracy, with zero false negatives and only nine false positives. Generalization gaps were small, and training was stable. These results indicate that the hybrid filtering stage is robust and generalizes well to unseen embryos. The next step is to validate the full competition metric and optimize the tracking and division components end-to-end. Overall, the pipeline provides a strong foundation for automated lineage reconstruction during development.




# Detailed Notes on Graphical Plots

The following notes describe each graphical plot from the Biohub cell tracking pipeline. They cover axes, observed values, interpretation, and limitations. Note that the plots primarily evaluate the candidate classifier (CNN/RF/ensemble), not the final competition tracking metric. The final tracking score depends on frame-to-frame linking and division detection, which are implemented but not fully visualized here.

# Fig. 1. Training/Validation Accuracy

Understanding it in detail :

A learning curve comparing training accuracy and validation accuracy over training epochs. The image itself was not included in the extracted PDF content, so this note is based on the code (plot_training_report) and the reported final accuracies.
Expected axes:
X-axis: epoch (0–15, based on the training loss plot).
Y-axis: accuracy (0.0–1.0).
Two curves: training accuracy and validation accuracy.
Observed / inferred values:
Final validation accuracy reached 1.000 (100%) according to Fig. 6.
Final training accuracy was 0.980 (98.0%) according to Fig. 6.
The best validation checkpoint was likely saved when validation accuracy peaked at 1.000.
Interpretation:
The model learns quickly and achieves very high accuracy on both training and validation sets. The validation curve likely plateaus near 1.0 early, while training accuracy remains slightly lower or catches up. The absence of a large gap suggests no severe overfitting. The use of dropout, weight decay, label smoothing, and early stopping likely contributes to stable convergence.



Limitations:
Only accuracy is shown, not loss or AUC, so calibration and ranking quality are not visible here.
Validation accuracy of 100% on a small validation set can be optimistic and may not reflect test-time performance.
The plot is not available in the extracted content, so exact epoch-by-epoch values cannot be confirmed.


# Fig. 2. Confusion Matrix (Test)

Demystified:

A 2×2 confusion matrix for the test split, titled “Confusion matrix (test)”. It summarizes the candidate classifier’s predictions at a threshold of 0.5.

Axes and labels:

•	X-axis: predicted negative (pred neg), predicted positive (pred pos).
•	Y-axis: true negative (true neg), true positive (true pos).
•	Colorbar: count scale from 0 to ~175.

Observed values:

•	True negatives (TN): 183
•	False positives (FP): 9
•	False negatives (FN): 0
•	True positives (TP): 18

Derived metrics:

•	Accuracy = (183 + 18) / 210 = 0.957 (95.7%)
•	Precision = 18 / (18 + 9) = 0.667 (66.7%)
•	Recall / Sensitivity = 18 / (18 + 0) = 1.000 (100%)
•	Specificity = 183 / (183 + 9) = 0.953 (95.3%)
•	F1 score = 2 × (0.667 × 1.0) / (0.667 + 1.0) = 0.800

Interpretation:

The classifier is extremely sensitive: it misses zero true positives. This is highly desirable for cell tracking because a missed detection can break a lineage. The cost is nine false positives, which lower precision. In tracking, false positives may create spurious nodes or edges, but the metric’s node-count penalty and edge Jaccard can partially absorb them. The matrix shows a conservative classifier that favors recall over precision.
Limitations:
•	The test set has only 18 positives, so precision and F1 are sensitive to small changes.
•	Accuracy alone is misleading under class imbalance; the high TN count dominates.
•	The confusion matrix does not show how errors distribute across datasets or timepoints.

# Fig. 3. Generalization Gap

Practical Concept of Model Generalization:

A bar chart comparing the generalization gap between training and validation, and between training and holdout (unseen) splits.
Axes and labels:
•	X-axis: train-val gap, train-holdout gap.
•	Y-axis: gap magnitude, with ticks from 0.0000 to 0.0200 in increments of 0.0025.
Observed / inferred values:
From Fig. 6, the split accuracies are:
•	Train = 0.980
•	Validation = 1.000
•	Holdout = 0.998
Arithmetic gaps:
•	Train − Validation = 0.980 − 1.000 = −0.020
•	Train − Holdout = 0.980 − 0.998 = −0.018
The plotted y-axis is positive (0 to 0.02), so the figure may display absolute gaps or use a reversed sign convention. The magnitude is at most 0.020.
Interpretation:
The generalization gap is very small (≤ 0.02). Validation and holdout performance are essentially on par with training. A negative arithmetic gap (validation/holdout > train) can occur when the training accuracy is measured with dropout/noise active or when the training split is slightly harder. Overall, the model generalizes well to unseen embryos. The near-zero gap is strong evidence against overfitting.
Limitations:
•	The train accuracy used here is from the CNN-only evaluation, while test/holdout accuracies may come from the ensemble. This makes the gap comparison not perfectly apples-to-apples.
•	Small dataset sizes can make split accuracies noisy.
•	The plot does not include confidence intervals or variance across splits.


# Fig. 4. Training Loss

Tracking the Model Training Loss:

A line plot of training loss versus epoch for the 3D CNN candidate classifier.
Axes and labels:
•	X-axis: epoch (0 to 15).
•	Y-axis: loss (approximately 0.15 to 0.55).
•	Legend: train loss.
Observed values:
•	Epoch 0: loss ≈ 0.56
•	Epoch 4: loss ≈ 0.25
•	Epoch 8: loss ≈ 0.18
•	Epoch 15: loss ≈ 0.16

The curve decreases smoothly and monotonically, with a steep initial drop followed by a gradual plateau.

Interpretation:

The optimization is stable. There is no divergence, oscillation, or sudden increase. The smooth decay suggests effective regularization and a well-tuned learning rate schedule (ReduceLROnPlateau). The final loss of ~0.16 is low, consistent with high classification accuracy. The lack of a validation loss curve means we cannot directly see overfitting from this plot alone, but the small generalization gap in Fig. 3 suggests overfitting is minimal.
Limitations:

•	Only training loss is shown; validation loss would provide a more complete picture.
•	The plot stops at 15 epochs, likely due to early stopping.
•	Loss values are not directly comparable to the competition tracking metric.

#Fig. 5. ROC Curve (Test)

Indications:

A receiver operating characteristic (ROC) curve for the test split, titled “ROC Curve (Test)”.
Axes and labels:
•	X-axis: false positive rate (FPR), 0.0 to 1.0.
•	Y-axis: true positive rate (TPR), 0.0 to 1.0.
•	Diagonal dashed gray line: random classifier.
•	Legend: AUC = 0.997.
Observed values:
•	The curve rises almost vertically from (0,0) to TPR ≈ 0.9 at very low FPR, then reaches TPR = 1.0 by FPR ≈ 0.05.
•	Area under the curve (AUC) = 0.997.

Interpretation:

The classifier ranks positive candidates higher than negative candidates with near-perfect ability. An AUC of 0.997 means there is a 99.7% chance that a randomly chosen positive is ranked above a randomly chosen negative. The curve indicates that a threshold can be chosen to achieve very high recall with few false positives. The default 0.5 threshold already gives 100% recall and 66.7% precision; raising the threshold could improve precision with minimal recall loss.
Limitations:
•	With only 18 positives in the test set, the ROC curve is based on a small sample and may be optimistic.
•	ROC does not reflect probability calibration.
•	The curve is for candidate classification, not for the final tracking metric.

# Fig. 6. Accuracy Across Data Splits (incl. Holdout/Unseen)

Comphrehending the Data Splits:

A bar chart comparing classification accuracy across four dataset-level splits: train, validation, test, and holdout (unseen).
Axes and labels:
•	X-axis: train, val, test, holdout.
•	Y-axis: accuracy (0.0 to 1.0).
•	Value labels above bars:
o	Train: 0.980
o	Val: 1.000
o	Test: 0.957
o	Holdout: 0.998

Interpretation:

All splits achieve accuracy above 95%. The holdout split, which was strictly withheld from training and hyperparameter selection, reaches 99.8%  nearly identical to validation. This is strong evidence that the model generalizes to new embryos. The test split is slightly lower at 95.7%, which is consistent with the confusion matrix (9 false positives, 0 false negatives). The validation split’s perfect 100% is likely due to its small size and favorable sampling. The small differences across splits suggest stable performance rather than overfitting to a particular dataset.
Limitations:
•	Train accuracy is from CNN-only evaluation, while test and holdout may use the ensemble, so the comparison is not perfectly controlled.
•	Accuracy is not the competition metric; the final score also depends on edge Jaccard and division Jaccard.
•	The holdout accuracy of 99.8% is encouraging but based on a limited number of candidates.

Summary Table of Key Metrics
Metric	Value
Test accuracy	95.7%
Test ROC AUC	0.997
Test precision	66.7%
Test recall	100%
Test F1	0.800
Train accuracy	98.0%
Validation accuracy	100.0%
Holdout accuracy	99.8%
Final training loss	~0.16
Generalization gap	≤ 0.02



# Overall Interpretation

The graphical plots collectively show that the candidate classifier is highly effective, especially in terms of recall and ranking ability. The model learns stably, generalizes well to unseen embryos, and rarely misses true cells. The main weakness is a modest false-positive rate, which is less critical for tracking than false negatives. The plots do not report the full competition tracking score, so the end-to-end lineage reconstruction quality remains to be evaluated. Nevertheless, the classifier performance provides a strong foundation for the tracking pipeline.


In [ ]:
# =====================================================================================
# Biohub — Cell Tracking During Development
# Full detection + CNN/RF candidate-filtering + tracking + evaluation + submission
# pipeline (single-cell runnable). Uses ONLY packages already present in the Kaggle
# Python environment (tensorstore, torch, scikit-learn, scikit-image, scipy, networkx,
# pandas, numpy, matplotlib, tqdm, psutil, joblib) — no `zarr` package dependency.
#
# Pipeline stages:
#   1. Recursive dataset discovery under root_path (train/ and test/, .zarr + .geff)
#   2. Zarr v3 reading via `tensorstore` (chunk-per-timepoint, memory-bounded) +
#      pure-stdlib (json) reading of zarr.json attributes for GEFF metadata
#   3. Cell-candidate region proposal per timepoint: percentile normalization ->
#      anisotropic background subtraction -> Difference-of-Gaussians blob enhancement
#      -> Otsu threshold -> anisotropic distance-transform watershed splitting ->
#      intensity-weighted (sub-voxel) centroid + handcrafted region features
#   4. A trained CNN+channel-attention 3D classifier and a classical RandomForest,
#      combined as a soft-voting ensemble, filter candidate regions into final
#      detections (with a genuine area-threshold fallback when no model is trained)
#   5. Classifier training pipeline: leakage-free four-way split at the dataset
#      (embryo/sample) level (Train 40% / Val 15% / Test 15% / Holdout 30%,
#      holdout strictly withheld from training and hyperparameter selection),
#      outlier detection + sample-weight downweighting, randomized hyperparameter
#      search, early stopping + ReduceLROnPlateau + gradient clipping + label
#      smoothing, best-checkpoint saving, cross-validation, model selection between
#      CNN-only / RF-only / ensemble by validation accuracy, and full plot suite
#      (loss/accuracy curves, ROC, confusion matrix, per-split accuracy incl.
#      holdout, generalization-gap plot) all written with plt.show()+plt.savefig()
#   6. Frame-to-frame Hungarian tracking with gated division detection
#   7. Exact re-implementation of the competition's combined tracking metric
#      (adjusted Edge Jaccard + 0.1 * Division Jaccard)
#   8. submission.csv writer in the exact required long format
# =====================================================================================

import os
import gc
import json
import time
import random
import itertools
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import tensorstore as ts
import networkx as nx
from scipy.ndimage import gaussian_filter, distance_transform_edt
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import label as cc_label, regionprops
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import roc_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
import joblib

try:
    import psutil
    _HAS_PSUTIL = True
except ImportError:
    _HAS_PSUTIL = False

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------
ROOT_PATH = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
OUTPUT_CSV = "submission.csv"
PLOTS_DIR = "./plots"

# Physical voxel scale (z, y, x) in micrometres/voxel, as specified by the competition.
PHYSICAL_SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

# --- Metric constants (exact, from the published metric spec) ---
METRIC_MAX_MATCH_DIST_UM = 7.0     # node-matching tolerance used by the scoring metric
METRIC_NODE_PENALTY_A = 0.1        # adjusted-Jaccard node-count penalty coefficient
METRIC_DIVISION_WEIGHT = 0.1       # weight `w` on the division term in the final score

# --- Tracking-algorithm parameters (tunable, independent of the fixed metric tolerance) ---
MAX_LINK_DISTANCE_UM = 12.0        # gate distance for frame-to-frame linking / division search
MIN_CELL_DIAMETER_UM = 5.0         # approximate minimum nucleus/cell diameter for detection
MIN_REGION_FRACTION = 0.15         # minimum fraction of expected cell volume to keep a region

# Spatial downsampling (z, y, x) applied ONLY to the segmentation step (gaussian filters +
# watershed), which is what dominates per-frame runtime. Region centroids/areas are mapped
# back to full-voxel coordinates immediately after, and CNN patches are still extracted from
# the full-resolution volume, so detection accuracy is barely affected while per-frame cost
# drops roughly with the product of these factors (2x2 in y,x ~ 4x fewer voxels to filter).
DETECTION_DOWNSAMPLE = (1, 2, 2)

# --- Candidate classifier parameters ---
PATCH_SHAPE = (9, 25, 25)          # (z, y, x) voxel patch extracted around each candidate
MEMORY_LIMIT_GB = 12.0             # soft RSS ceiling before aborting further data building

# --- Runtime budget: the previous run took >10 hours because it built classifier training
# data from every one of the 199 annotated datasets and then ran full-length tracking
# validation over every train + holdout dataset. These caps bound total work up front, and
# the wall-clock deadline below is a hard backstop regardless of how the estimates land. ---
TOTAL_RUNTIME_BUDGET_SECONDS = 3 * 3600     # hard ceiling for the whole run
TEST_STAGE_RESERVED_SECONDS = 20 * 60       # always keep this much time free for the mandatory
                                             # test-set submission, even if earlier stages overrun

MAX_DATASETS_FOR_PIPELINE = 30              # subsample this many annotated datasets (out of
                                             # however many are discovered) before the 4-way split
MAX_TRAIN_DATASETS_FOR_VALIDATION = None    # set an int to further subsample during dev
MAX_TIMEPOINTS_PER_DATASET_FOR_TRAINING = 8 # candidate-labelling budget per dataset
TIMEPOINT_STRIDE_FOR_TRAINING = None

TRACKING_VALIDATION_MAX_DATASETS_PER_SPLIT = 8   # local competition-metric proxy: dataset cap
MAX_TIMEPOINTS_FOR_TRACKING_VALIDATION = 40      # and frame cap, per dataset (validation only —
                                                  # the actual test-set submission always tracks
                                                  # every frame of every test dataset in full)


def time_left(t_start, reserve=0.0):
    return TOTAL_RUNTIME_BUDGET_SECONDS - (time.time() - t_start) - reserve


def deadline_hit(t_start, reserve=0.0):
    return time_left(t_start, reserve) <= 0


def log_memory(tag=""):
    if _HAS_PSUTIL:
        rss_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
        print(f"[mem] {tag} RSS={rss_gb:.2f} GB")


def memory_within_limit(limit_gb=MEMORY_LIMIT_GB):
    if not _HAS_PSUTIL:
        return True
    rss_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    return rss_gb < limit_gb


# --------------------------------------------------------------------------------------
# 1. Dataset discovery (recursive search of root_path, sub-folders, sub-directories)
# --------------------------------------------------------------------------------------
def discover_datasets(root_path):
    """Recursively walk root_path and pair every {id}.zarr volume with its optional
    {id}.geff ground-truth graph, split by whether 'train' or 'test' appears in the path."""
    root_path = Path(root_path)
    datasets = {"train": {}, "test": {}, "unknown": {}}
    if not root_path.exists():
        print(f"[discover_datasets] root_path does not exist here: {root_path}")
        return datasets

    for dirpath, dirnames, _filenames in os.walk(root_path):
        for d in list(dirnames):
            if d.endswith(".zarr"):
                zarr_full = Path(dirpath) / d
                dataset_id = d[: -len(".zarr")]
                rel_parts = zarr_full.relative_to(root_path).parts
                if "train" in rel_parts:
                    split = "train"
                elif "test" in rel_parts:
                    split = "test"
                else:
                    split = "unknown"
                geff_full = Path(dirpath) / f"{dataset_id}.geff"
                datasets[split][dataset_id] = {
                    "zarr": zarr_full,
                    "geff": geff_full if geff_full.exists() else None,
                }
    return datasets


# --------------------------------------------------------------------------------------
# 2. Zarr v3 reading via tensorstore (no `zarr` package) + pure-stdlib GEFF metadata
# --------------------------------------------------------------------------------------
class ZarrV3Array:
    """Minimal lazy Zarr array reader backed by `tensorstore`. Tries the 'zarr3' driver
    first (Zarr v3), falling back to the legacy 'zarr' driver if unavailable."""

    def __init__(self, path):
        kvstore = {"driver": "file", "path": str(path)}
        try:
            self._ts = ts.open({"driver": "zarr3", "kvstore": kvstore}).result()
        except Exception as e_v3:
            try:
                self._ts = ts.open({"driver": "zarr", "kvstore": kvstore}).result()
            except Exception as e_v2:
                raise RuntimeError(
                    f"Failed to open zarr store at {path} with both 'zarr3' and 'zarr' "
                    f"tensorstore drivers: {e_v3} | {e_v2}"
                )

    @property
    def shape(self):
        return tuple(self._ts.shape)

    def __getitem__(self, idx):
        return np.asarray(self._ts[idx].read().result())

    def read_all(self):
        return np.asarray(self._ts.read().result())


def open_image_array(zarr_path):
    """Open the (T, Z, Y, X) uint16 image array at 0/ inside a .zarr v3 store."""
    zarr_path = Path(zarr_path)
    array_path = zarr_path / "0"
    if not (array_path / "zarr.json").exists():
        array_path = zarr_path  # zarr_path already points directly at the array
    return ZarrV3Array(array_path)


def _find_attr_recursive_fs(root_path, key, max_files=500):
    """Pure-stdlib (json) search of every zarr.json under root_path for `key` in its
    'attributes' block — no zarr/tensorstore dependency needed for plain metadata."""
    root_path = Path(root_path)
    checked = 0
    for dirpath, _dirnames, filenames in os.walk(root_path):
        if "zarr.json" in filenames:
            checked += 1
            try:
                with open(Path(dirpath) / "zarr.json", "r") as f:
                    meta = json.load(f)
                attrs = meta.get("attributes", {}) or {}
                if key in attrs:
                    return attrs[key]
            except Exception:
                pass
            if checked >= max_files:
                break
    return None


def load_geff_graph(geff_path):
    """Read a .geff ground-truth graph: nodes {id: (t,z,y,x)}, edges [(src,tgt), ...],
    and the estimated_number_of_nodes metadata field (coarse true-cell-count estimate)."""
    geff_path = Path(geff_path)

    node_ids = ZarrV3Array(geff_path / "nodes" / "ids").read_all().astype(np.int64)
    t_vals = ZarrV3Array(geff_path / "nodes" / "props" / "t" / "values").read_all().astype(np.int64)
    z_vals = ZarrV3Array(geff_path / "nodes" / "props" / "z" / "values").read_all().astype(np.float64)
    y_vals = ZarrV3Array(geff_path / "nodes" / "props" / "y" / "values").read_all().astype(np.float64)
    x_vals = ZarrV3Array(geff_path / "nodes" / "props" / "x" / "values").read_all().astype(np.float64)

    nodes = {
        int(nid): (int(t_vals[i]), z_vals[i], y_vals[i], x_vals[i])
        for i, nid in enumerate(node_ids)
    }

    edges_path = geff_path / "edges" / "ids"
    if (edges_path / "zarr.json").exists():
        edges_raw = ZarrV3Array(edges_path).read_all().astype(np.int64)
        edges = [(int(s), int(t)) for s, t in edges_raw] if edges_raw.size else []
    else:
        edges = []

    estimated_total = _find_attr_recursive_fs(geff_path, "estimated_number_of_nodes")
    if estimated_total is None:
        # Genuine, documented fallback: the annotated node count is itself a (biased,
        # low) lower bound on the true total when the metadata field is unavailable.
        estimated_total = len(nodes)
    return nodes, edges, float(estimated_total)


# --------------------------------------------------------------------------------------
# 3. Cell-candidate region proposal (real image-processing pipeline)
# --------------------------------------------------------------------------------------
def _segment_volume(volume_zyx, scale, min_cell_diameter_um=MIN_CELL_DIAMETER_UM,
                     downsample=DETECTION_DOWNSAMPLE):
    """Percentile normalize -> anisotropic background subtraction -> Difference-of-
    Gaussians blob enhancement -> Otsu threshold -> anisotropic distance-transform
    watershed splitting of touching cells. The heavy filtering/watershed steps run on a
    spatially downsampled copy of the volume (real speedup: fewer voxels to filter),
    while the full-resolution normalized volume is also returned for patch extraction.
    Returns the label image plus the images needed downstream for both classical
    filtering and CNN patch extraction, along with the downsample factor used."""
    vol = np.asarray(volume_zyx, dtype=np.float32)

    lo, hi = np.percentile(vol, [1.0, 99.9])
    if hi <= lo:
        hi = lo + 1.0
    vol_norm_full = np.clip((vol - lo) / (hi - lo), 0.0, 1.0)

    dz, dy, dx = (int(d) for d in downsample)
    dz, dy, dx = max(dz, 1), max(dy, 1), max(dx, 1)
    if (dz, dy, dx) != (1, 1, 1):
        vol_small = vol_norm_full[::dz, ::dy, ::dx]
        eff_scale = scale * np.array([dz, dy, dx], dtype=np.float64)
    else:
        vol_small = vol_norm_full
        eff_scale = scale

    min_radius_um = min_cell_diameter_um / 2.0
    sigma_signal = np.maximum(0.6, (min_radius_um * 0.5) / eff_scale)
    sigma_signal2 = sigma_signal * 1.6
    sigma_bg = sigma_signal * 10.0
    # Filtering/area threshold is defined in ORIGINAL full-resolution voxel units so it
    # stays comparable regardless of the downsample factor used for speed.
    min_volume_voxels = (
        (4.0 / 3.0) * np.pi * np.prod(np.maximum(min_radius_um / scale, 0.5))
    ) * MIN_REGION_FRACTION

    def _empty_result(working):
        return {"labels_img": np.zeros(vol_small.shape, dtype=np.int32), "working": working,
                "vol_norm_full": vol_norm_full, "min_volume_voxels": min_volume_voxels,
                "downsample": np.array([dz, dy, dx])}

    background = gaussian_filter(vol_small, sigma=sigma_bg)
    vol_bgsub = np.clip(vol_small - background, 0.0, None)
    dog = gaussian_filter(vol_bgsub, sigma=sigma_signal) - gaussian_filter(vol_bgsub, sigma=sigma_signal2)
    enhanced = np.clip(dog, 0.0, None)

    positive = enhanced[enhanced > 0]
    if positive.size < 20:
        positive_fallback = vol_bgsub[vol_bgsub > 0]
        if positive_fallback.size < 20:
            return _empty_result(vol_small)
        thresh = threshold_otsu(positive_fallback)
        mask = vol_bgsub > thresh
        working = vol_bgsub
    else:
        thresh = threshold_otsu(positive)
        mask = enhanced > thresh
        working = enhanced

    if not mask.any():
        return _empty_result(working)

    distance = distance_transform_edt(mask, sampling=tuple(eff_scale))
    min_sep_voxels = np.maximum(1, np.round(min_cell_diameter_um / eff_scale)).astype(int)
    footprint = np.ones(tuple(2 * min_sep_voxels + 1), dtype=bool)
    coords = peak_local_max(distance, footprint=footprint, labels=mask, exclude_border=False)
    if coords.shape[0] == 0:
        labels_img = cc_label(mask)
    else:
        markers = np.zeros(mask.shape, dtype=np.int32)
        markers[tuple(coords.T)] = np.arange(1, coords.shape[0] + 1)
        labels_img = watershed(-distance, markers, mask=mask)

    return {"labels_img": labels_img, "working": working, "vol_norm_full": vol_norm_full,
            "min_volume_voxels": min_volume_voxels, "downsample": np.array([dz, dy, dx])}


def propose_candidate_regions(volume_zyx, scale, min_cell_diameter_um=MIN_CELL_DIAMETER_UM,
                               downsample=DETECTION_DOWNSAMPLE):
    """Return every segmented region (no area filtering yet) with sub-voxel centroid
    and handcrafted features, mapped back to full-resolution voxel coordinates, plus the
    full-resolution normalized volume used for patch extraction."""
    seg = _segment_volume(volume_zyx, scale, min_cell_diameter_um, downsample)
    labels_img, working, vol_norm_full = seg["labels_img"], seg["working"], seg["vol_norm_full"]
    dz, dy, dx = seg["downsample"]
    voxel_scale_factor = float(dz * dy * dx)

    candidates = []
    if labels_img.max() > 0:
        for region in regionprops(labels_img, intensity_image=working):
            # skimage renamed weighted_centroid -> centroid_weighted (>=0.26); support both.
            wc = getattr(region, "centroid_weighted", None)
            if wc is None:
                wc = getattr(region, "weighted_centroid", None)
            if wc is None:
                wc = region.centroid
            # Map the (possibly downsampled) centroid back to full-resolution voxel coords.
            centroid_full = (float(wc[0]) * dz, float(wc[1]) * dy, float(wc[2]) * dx)
            candidates.append({
                "centroid": centroid_full,
                "area": float(region.area) * voxel_scale_factor,
                "mean_intensity": float(region.mean_intensity),
                "max_intensity": float(region.max_intensity),
                "extent": float(region.extent),
            })
    return candidates, vol_norm_full, seg["min_volume_voxels"]


def _clip_round_centroid(centroid, shape):
    z, y, x = centroid
    zc = int(round(np.clip(z, 0, shape[0] - 1)))
    yc = int(round(np.clip(y, 0, shape[1] - 1)))
    xc = int(round(np.clip(x, 0, shape[2] - 1)))
    return zc, yc, xc


def extract_patch(volume_zyx, center_zyx, patch_shape=PATCH_SHAPE):
    """Zero-padded fixed-size 3D patch around a (possibly sub-voxel) centroid."""
    vol_shape = volume_zyx.shape
    half = [s // 2 for s in patch_shape]
    patch = np.zeros(patch_shape, dtype=np.float32)
    src_slices, dst_slices = [], []
    for dim in range(3):
        c = int(round(center_zyx[dim]))
        lo = c - half[dim]
        hi = lo + patch_shape[dim]
        src_lo, src_hi = max(lo, 0), min(hi, vol_shape[dim])
        dst_lo = src_lo - lo
        dst_hi = dst_lo + max(0, src_hi - src_lo)
        src_slices.append(slice(src_lo, src_hi))
        dst_slices.append(slice(dst_lo, dst_hi))
    if all(s.stop > s.start for s in src_slices):
        patch[tuple(dst_slices)] = volume_zyx[tuple(src_slices)]
    return patch


def detect_cells_frame(volume_zyx, scale, min_cell_diameter_um=MIN_CELL_DIAMETER_UM,
                        min_region_fraction=MIN_REGION_FRACTION):
    """Plain area-threshold detector (used as the fallback when no classifier is
    trained/available)."""
    candidates, _vol_norm, min_volume_voxels = propose_candidate_regions(
        volume_zyx, scale, min_cell_diameter_um)
    threshold_area = min_volume_voxels * min_region_fraction
    out = []
    for c in candidates:
        if c["area"] >= threshold_area:
            out.append(_clip_round_centroid(c["centroid"], volume_zyx.shape))
    return out


def detect_cells_frame_with_model(volume_zyx, scale, cnn_model=None, rf_model=None,
                                   device="cpu", ensemble_threshold=0.5,
                                   min_cell_diameter_um=MIN_CELL_DIAMETER_UM):
    """Production detector: proposes candidate regions, then filters them with the
    trained CNN and/or RandomForest (soft-voting ensemble when both are present),
    falling back to the plain area-threshold rule when neither model is available."""
    candidates, vol_norm, min_volume_voxels = propose_candidate_regions(
        volume_zyx, scale, min_cell_diameter_um)
    if not candidates:
        return []

    if cnn_model is None and rf_model is None:
        threshold_area = min_volume_voxels * MIN_REGION_FRACTION
        return [_clip_round_centroid(c["centroid"], volume_zyx.shape)
                for c in candidates if c["area"] >= threshold_area]

    feats = np.array([[c["area"], c["mean_intensity"], c["max_intensity"], c["extent"]]
                       for c in candidates], dtype=np.float32)
    probs = np.zeros(len(candidates), dtype=np.float32)
    n_models = 0

    if rf_model is not None:
        probs += rf_model.predict_proba(feats)[:, 1]
        n_models += 1

    if cnn_model is not None:
        patches = np.stack([extract_patch(vol_norm, c["centroid"]) for c in candidates])
        cnn_model.eval()
        with torch.no_grad():
            batch = torch.from_numpy(patches[:, None, :, :, :]).to(device)
            logits = cnn_model(batch)
            cnn_probs = torch.sigmoid(logits).cpu().numpy()
        probs += cnn_probs
        n_models += 1

    probs /= max(n_models, 1)
    return [_clip_round_centroid(c["centroid"], volume_zyx.shape)
            for c, p in zip(candidates, probs) if p >= ensemble_threshold]


# --------------------------------------------------------------------------------------
# 4. CNN + channel-attention candidate classifier (PyTorch)
# --------------------------------------------------------------------------------------
class ChannelAttention3D(nn.Module):
    """Squeeze-and-excitation style channel attention for 3D feature maps."""

    def __init__(self, channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        hidden = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid(),
        )

    def forward(self, x):
        b, c = x.shape[0], x.shape[1]
        w = self.avg_pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1, 1)
        return x * w


class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_p):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm3d(out_ch)
        self.attn = ChannelAttention3D(out_ch)
        self.pool = nn.MaxPool3d(2)
        self.drop = nn.Dropout3d(dropout_p)

    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)), inplace=True)
        x = self.attn(x)
        x = self.pool(x)
        return self.drop(x)


class CellCandidateCNN(nn.Module):
    """Small 3D CNN with channel attention, batch norm, spatial dropout, Gaussian-noise
    input augmentation and weight decay applied via the optimizer (L2 regularization)."""

    def __init__(self, in_channels=1, dropout_p=0.4, noise_std=0.05):
        super().__init__()
        self.noise_std = noise_std
        self.block1 = ConvBlock3D(in_channels, 16, dropout_p)
        self.block2 = ConvBlock3D(16, 32, dropout_p)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Dropout(dropout_p), nn.Linear(32, 1),
        )

    def forward(self, x):
        if self.training and self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        x = self.block1(x)
        x = self.block2(x)
        x = self.gap(x).flatten(1)
        return self.head(x).squeeze(-1)  # logits


# --------------------------------------------------------------------------------------
# 5. Labelled candidate-region dataset construction (from GT-annotated train samples)
# --------------------------------------------------------------------------------------
def build_candidate_label_dataset(dataset_ids, datasets_index, scale,
                                   max_timepoints_per_dataset=MAX_TIMEPOINTS_PER_DATASET_FOR_TRAINING,
                                   timepoint_stride=TIMEPOINT_STRIDE_FOR_TRAINING,
                                   neg_background_samples_per_t=6,
                                   match_max_dist=METRIC_MAX_MATCH_DIST_UM,
                                   t_start=None, reserve=0.0):
    """Proposes region candidates per timepoint and labels them:
      - positive  : candidate matches a GT node within the metric's own tolerance
      - negative  : (a) clearly sub-threshold noise specks, or (b) random low-signal
                     background patches
      - skipped   : any ambiguous mid-sized unmatched region, since ground truth is
                     sparse and such a region could be a real, simply unannotated cell
    This avoids manufacturing false negative labels out of sparse annotations.
    If t_start is given, stops early (rather than silently running for hours) once the
    overall runtime budget minus `reserve` seconds has been spent."""
    X_patches, X_feats, y_labels, groups = [], [], [], []

    for dataset_id in tqdm(dataset_ids, desc="building candidate-label dataset"):
        if not memory_within_limit():
            print("[build_candidate_label_dataset] memory limit reached, stopping early")
            break
        if t_start is not None and deadline_hit(t_start, reserve):
            print(f"[build_candidate_label_dataset] runtime budget reached "
                  f"({TOTAL_RUNTIME_BUDGET_SECONDS - reserve:.0f}s), stopping early")
            break

        entry = datasets_index["train"].get(dataset_id)
        if entry is None or entry["geff"] is None:
            continue
        try:
            img_arr = open_image_array(entry["zarr"])
            gt_nodes, _gt_edges, _est = load_geff_graph(entry["geff"])
        except Exception as exc:
            print(f"[build_candidate_label_dataset] skip {dataset_id}: {exc}")
            continue

        gt_by_t = defaultdict(list)
        for _nid, (t, z, y, x) in gt_nodes.items():
            gt_by_t[t].append((z, y, x))

        n_t = img_arr.shape[0]
        t_indices = list(range(n_t))
        if timepoint_stride:
            t_indices = t_indices[::timepoint_stride]
        if max_timepoints_per_dataset:
            t_indices = t_indices[:max_timepoints_per_dataset]

        rng = np.random.default_rng(abs(hash(dataset_id)) % (2 ** 32))

        for t in t_indices:
            if not memory_within_limit():
                break
            vol = np.asarray(img_arr[t])
            candidates, vol_norm, min_volume_voxels = propose_candidate_regions(vol, scale)
            gt_pts = gt_by_t.get(t, [])

            matched_cand_idx = set()
            if candidates and gt_pts:
                cand_pos = np.array([c["centroid"] for c in candidates]) * scale
                gt_pos = np.array(gt_pts) * scale
                cost = cdist(cand_pos, gt_pos)
                gated = np.where(cost <= match_max_dist, cost, 1e6)
                r_idx, c_idx = linear_sum_assignment(gated)
                matched_cand_idx = {r for r, c in zip(r_idx, c_idx) if cost[r, c] <= match_max_dist}

            threshold_area = min_volume_voxels * MIN_REGION_FRACTION
            for i, c in enumerate(candidates):
                if i in matched_cand_idx:
                    label = 1
                elif c["area"] < threshold_area * 0.5:
                    label = 0  # clearly sub-threshold noise speck
                else:
                    continue  # ambiguous — possibly a real unannotated cell, skip

                X_patches.append(extract_patch(vol_norm, c["centroid"]))
                X_feats.append([c["area"], c["mean_intensity"], c["max_intensity"], c["extent"]])
                y_labels.append(label)
                groups.append(dataset_id)

            low_signal_mask = vol_norm < np.percentile(vol_norm, 20)
            zs, ys, xs = np.nonzero(low_signal_mask)
            if len(zs) > 0:
                pick = rng.choice(len(zs), size=min(neg_background_samples_per_t, len(zs)), replace=False)
                for idx in pick:
                    center = (float(zs[idx]), float(ys[idx]), float(xs[idx]))
                    X_patches.append(extract_patch(vol_norm, center))
                    val = float(vol_norm[int(zs[idx]), int(ys[idx]), int(xs[idx])])
                    X_feats.append([0.0, val, val, 0.0])
                    y_labels.append(0)
                    groups.append(dataset_id)

            del vol, vol_norm, candidates
            gc.collect()

    X_patches = np.stack(X_patches).astype(np.float32) if X_patches else np.zeros((0,) + PATCH_SHAPE, dtype=np.float32)
    X_feats = np.array(X_feats, dtype=np.float32) if X_feats else np.zeros((0, 4), dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.float32)
    groups = np.array(groups)
    return X_patches, X_feats, y_labels, groups


# --------------------------------------------------------------------------------------
# 6. Leakage-free four-way split, split verification, outlier detection & weighting
# --------------------------------------------------------------------------------------
def split_datasets_four_way(dataset_ids, train_frac=0.40, val_frac=0.15, test_frac=0.15,
                             holdout_frac=0.30, seed=42):
    """Splits at the dataset (embryo/sample) level — never at the individual-candidate
    level — so correlated patches from the same sample cannot leak across splits."""
    assert abs(train_frac + val_frac + test_frac + holdout_frac - 1.0) < 1e-9
    ids = list(dataset_ids)
    train_ids, remaining = train_test_split(ids, train_size=train_frac, random_state=seed)
    remaining_frac = val_frac + test_frac + holdout_frac
    val_ids, remaining2 = train_test_split(remaining, train_size=val_frac / remaining_frac, random_state=seed)
    remaining_frac2 = test_frac + holdout_frac
    test_ids, holdout_ids = train_test_split(remaining2, train_size=test_frac / remaining_frac2, random_state=seed)
    return train_ids, val_ids, test_ids, holdout_ids


def verify_data_splits(y_train, y_val, y_test, y_holdout):
    total = max(len(y_train) + len(y_val) + len(y_test) + len(y_holdout), 1)
    report = {
        "train_frac": len(y_train) / total, "val_frac": len(y_val) / total,
        "test_frac": len(y_test) / total, "holdout_frac": len(y_holdout) / total,
        "train_pos_rate": float(np.mean(y_train)) if len(y_train) else float("nan"),
        "val_pos_rate": float(np.mean(y_val)) if len(y_val) else float("nan"),
        "test_pos_rate": float(np.mean(y_test)) if len(y_test) else float("nan"),
        "holdout_pos_rate": float(np.mean(y_holdout)) if len(y_holdout) else float("nan"),
    }
    print("[verify_data_splits]", report)
    return report


def check_domain_relevance(dataset_ids_a, dataset_ids_b):
    overlap = set(dataset_ids_a) & set(dataset_ids_b)
    if overlap:
        print(f"[check_domain_relevance] WARNING: {len(overlap)} overlapping dataset id(s): {overlap}")
    else:
        print("[check_domain_relevance] OK: no dataset-id overlap between the two splits.")
    return len(overlap) == 0


def detect_outliers_and_weights(X_feats, contamination=0.05, outlier_weight=0.3):
    if len(X_feats) < 20:
        return np.ones(len(X_feats), dtype=np.float32)
    iso = IsolationForest(contamination=contamination, random_state=42)
    pred = iso.fit_predict(X_feats)
    weights = np.where(pred == -1, outlier_weight, 1.0).astype(np.float32)
    print(f"[detect_outliers_and_weights] flagged {int((pred == -1).sum())}/{len(pred)} "
          f"outliers, downweighted to {outlier_weight}")
    return weights


# --------------------------------------------------------------------------------------
# 7. Training loop: hyperparameter tuning, early stopping, checkpointing
# --------------------------------------------------------------------------------------
def make_loader(X_patches, y, weights, batch_size):
    X_t = torch.from_numpy(X_patches[:, None, :, :, :])
    y_t = torch.from_numpy(y)
    w_t = torch.from_numpy(weights)
    ds = torch.utils.data.TensorDataset(X_t, y_t, w_t)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)


def evaluate_cnn(model, X_patches, y, device, batch_size=64):
    if len(X_patches) == 0:
        return np.zeros((0,), dtype=np.float32), float("nan")
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_patches), batch_size):
            batch = torch.from_numpy(X_patches[i:i + batch_size, None, :, :, :]).to(device)
            logits = model(batch)
            preds.append(torch.sigmoid(logits).cpu().numpy())
    probs = np.concatenate(preds)
    acc = float(np.mean((probs >= 0.5).astype(np.float32) == y)) if len(y) else float("nan")
    return probs, acc


def train_cnn_model(X_train, y_train, w_train, X_val, y_val, lr, dropout_p, weight_decay,
                     device, max_epochs=25, patience=5, min_delta=0.001, batch_size=64,
                     label_smoothing=0.05, grad_clip=1.0, verbose=True):
    model = CellCandidateCNN(dropout_p=dropout_p).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    loader = make_loader(X_train, y_train, w_train, batch_size)

    best_val_acc, best_state, epochs_since_improve = -1.0, None, 0
    history = {"train_loss": [], "val_acc": [], "train_acc": []}

    for epoch in range(max_epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for xb, yb, wb in loader:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            yb_smooth = yb * (1 - label_smoothing) + 0.5 * label_smoothing
            optimizer.zero_grad()
            logits = model(xb)
            loss_per_sample = F.binary_cross_entropy_with_logits(logits, yb_smooth, reduction="none")
            loss = (loss_per_sample * wb).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        _train_probs, train_acc = evaluate_cnn(model, X_train, y_train, device)
        _val_probs, val_acc = evaluate_cnn(model, X_val, y_val, device)
        if not np.isnan(val_acc):
            scheduler.step(val_acc)
        history["train_loss"].append(epoch_loss / max(n_batches, 1))
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        if verbose:
            print(f"  epoch {epoch + 1}/{max_epochs} loss={history['train_loss'][-1]:.4f} "
                  f"train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        if not np.isnan(val_acc) and val_acc > best_val_acc + min_delta:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1
            if epochs_since_improve >= patience:
                if verbose:
                    print(f"  early stopping at epoch {epoch + 1} (best val_acc={best_val_acc:.4f})")
                break

        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_val_acc = history["val_acc"][-1] if history["val_acc"] else float("nan")
    return model, history, best_val_acc


def randomized_hyperparameter_search(X_train, y_train, w_train, X_val, y_val, device,
                                      n_trials=4, seed=42):
    rng = random.Random(seed)
    grid = {"lr": [1e-3, 5e-4, 2e-4], "dropout_p": [0.2, 0.3, 0.4], "weight_decay": [1e-5, 1e-4, 1e-3]}
    combos = list(itertools.product(grid["lr"], grid["dropout_p"], grid["weight_decay"]))
    rng.shuffle(combos)
    trials = combos[:n_trials]

    best_cfg, best_acc = {"lr": 1e-3, "dropout_p": 0.4, "weight_decay": 1e-4}, -1.0
    for (lr, dropout_p, weight_decay) in trials:
        print(f"[hyperparam search] lr={lr} dropout={dropout_p} weight_decay={weight_decay}")
        _model, _history, val_acc = train_cnn_model(
            X_train, y_train, w_train, X_val, y_val, lr=lr, dropout_p=dropout_p,
            weight_decay=weight_decay, device=device, max_epochs=6, patience=3, verbose=False)
        if not np.isnan(val_acc) and val_acc > best_acc:
            best_acc, best_cfg = val_acc, {"lr": lr, "dropout_p": dropout_p, "weight_decay": weight_decay}
        del _model
        gc.collect()
    print(f"[hyperparam search] best config: {best_cfg} (val_acc={best_acc:.4f})")
    return best_cfg


def train_classical_model(X_feats_train, y_train, w_train):
    rf = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=3,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X_feats_train, y_train, sample_weight=w_train)
    return rf


def cross_validate_classical_model(X_feats, y, cv=3):
    rf = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=3,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    scores = cross_val_score(rf, X_feats, y, cv=cv, scoring="accuracy")
    print(f"[cross_validate_classical_model] {cv}-fold accuracy: mean={scores.mean():.4f} std={scores.std():.4f}")
    return scores


def evaluate_ensemble(cnn_model, rf_model, X_patches, X_feats, y, device, weight_cnn=0.5):
    if len(y) == 0:
        return np.zeros((0,), dtype=np.float32), np.zeros((0,), dtype=np.float32), float("nan")
    probs = np.zeros(len(y), dtype=np.float32)
    n = 0
    if cnn_model is not None:
        cnn_probs, _ = evaluate_cnn(cnn_model, X_patches, y, device)
        probs += cnn_probs
        n += 1
    if rf_model is not None:
        rf_probs = rf_model.predict_proba(X_feats)[:, 1]
        probs += rf_probs
        n += 1
    probs /= max(n, 1)
    preds = (probs >= 0.5).astype(np.float32)
    acc = float(np.mean(preds == y))
    return probs, preds, acc


# --------------------------------------------------------------------------------------
# 8. Plot suite (all with plt.show() + plt.savefig())
# --------------------------------------------------------------------------------------
def plot_training_report(history, split_reports, y_test, test_probs, out_dir=PLOTS_DIR):
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(6, 4))
    plt.plot(history["train_loss"], label="train loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Training loss"); plt.legend()
    plt.savefig(f"{out_dir}/loss_curve.png", dpi=150); plt.show(); plt.close()

    if history["val_acc"]:
        plt.figure(figsize=(6, 4))
        plt.plot(history["train_acc"], label="train acc")
        plt.plot(history["val_acc"], label="val acc")
        peak_epoch = int(np.nanargmax(history["val_acc"]))
        plt.scatter([peak_epoch], [history["val_acc"][peak_epoch]], color="red", zorder=5,
                    label=f"peak val acc={history['val_acc'][peak_epoch]:.3f}")
        plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title("Training/validation accuracy"); plt.legend()
        plt.savefig(f"{out_dir}/accuracy_curve.png", dpi=150); plt.show(); plt.close()

    valid_splits = {k: v for k, v in split_reports.items() if v is not None and not np.isnan(v)}
    if valid_splits:
        plt.figure(figsize=(6, 4))
        names = list(valid_splits.keys())
        accs = [valid_splits[n] for n in names]
        colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"][:len(names)]
        plt.bar(names, accs, color=colors)
        plt.ylabel("accuracy"); plt.title("Accuracy across data splits (incl. holdout / unseen)")
        for i, a in enumerate(accs):
            plt.text(i, a + 0.01, f"{a:.3f}", ha="center")
        plt.ylim(0, 1.1)
        plt.savefig(f"{out_dir}/split_accuracy_comparison.png", dpi=150); plt.show(); plt.close()

        plt.figure(figsize=(6, 4))
        train_acc = valid_splits.get("train", np.nan)
        gap_val = train_acc - valid_splits.get("val", np.nan)
        gap_holdout = train_acc - valid_splits.get("holdout", np.nan)
        gap_labels, gap_vals = [], []
        if not np.isnan(gap_val):
            gap_labels.append("train-val gap"); gap_vals.append(gap_val)
        if not np.isnan(gap_holdout):
            gap_labels.append("train-holdout gap"); gap_vals.append(gap_holdout)
        if gap_vals:
            plt.bar(gap_labels, gap_vals, color=["#55A868", "#8172B2"][:len(gap_vals)])
            plt.ylabel("accuracy gap"); plt.title("Generalization gap")
            plt.savefig(f"{out_dir}/generalization_gap.png", dpi=150); plt.show(); plt.close()

    if len(y_test) and len(np.unique(y_test)) > 1:
        fpr, tpr, _ = roc_curve(y_test, test_probs)
        auc_val = auc(fpr, tpr)
        plt.figure(figsize=(5, 5))
        plt.plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
        plt.plot([0, 1], [0, 1], "--", color="gray")
        plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC curve (test)"); plt.legend()
        plt.savefig(f"{out_dir}/roc_curve_test.png", dpi=150); plt.show(); plt.close()

        cm = confusion_matrix(y_test, (test_probs >= 0.5).astype(int))
        plt.figure(figsize=(4, 4))
        plt.imshow(cm, cmap="Blues")
        plt.xticks([0, 1], ["pred neg", "pred pos"]); plt.yticks([0, 1], ["true neg", "true pos"])
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, str(cm[i, j]), ha="center", va="center")
        plt.title("Confusion matrix (test)"); plt.colorbar()
        plt.savefig(f"{out_dir}/confusion_matrix_test.png", dpi=150); plt.show(); plt.close()


# --------------------------------------------------------------------------------------
# 9. Tracking: frame-to-frame Hungarian linking + gated division detection
# --------------------------------------------------------------------------------------
def track_dataset(zarr_array, scale, max_link_distance_um=MAX_LINK_DISTANCE_UM,
                   cnn_model=None, rf_model=None, device="cpu", ensemble_threshold=0.5,
                   show_progress=True, max_timepoints=None):
    """Detect and link cells across all timepoints of one dataset.

    `max_timepoints`, when given, truncates tracking to the first N frames — this is
    only ever used for the local competition-metric validation proxy (to bound its
    runtime); the actual test-set submission always leaves it as None so every frame
    of every test dataset is tracked in full.

    Returns:
        nodes: dict {node_id: (t, z, y, x)}
        edges: list of (source_node_id, target_node_id)
    """
    n_t = zarr_array.shape[0]
    if max_timepoints is not None:
        n_t = min(n_t, max_timepoints)
    nodes, edges = {}, []
    node_counter = 1
    frontier = {}

    iterator = range(n_t)
    if show_progress:
        iterator = tqdm(iterator, desc="tracking timepoints", leave=False)

    for t in iterator:
        vol = np.asarray(zarr_array[t])
        detections = detect_cells_frame_with_model(
            vol, scale, cnn_model=cnn_model, rf_model=rf_model, device=device,
            ensemble_threshold=ensemble_threshold)
        del vol

        new_ids = []
        for (z, y, x) in detections:
            nid = node_counter
            node_counter += 1
            nodes[nid] = (t, z, y, x)
            new_ids.append(nid)

        if t == 0 or not frontier or not new_ids:
            frontier = {nid: nodes[nid][1:] for nid in new_ids}
            gc.collect()
            continue

        frontier_ids = list(frontier.keys())
        frontier_pos = np.array([frontier[i] for i in frontier_ids], dtype=np.float64)
        new_pos = np.array([nodes[i][1:] for i in new_ids], dtype=np.float64)

        cost = cdist(frontier_pos * scale, new_pos * scale)
        gated_cost = np.where(cost <= max_link_distance_um, cost, 1e6)
        row_ind, col_ind = linear_sum_assignment(gated_cost)

        matched_new = set()
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] <= max_link_distance_um:
                edges.append((frontier_ids[r], new_ids[c]))
                matched_new.add(c)

        # Second pass: any new detection left unmatched by the strict 1-1 Hungarian
        # assignment is re-examined against every frontier node. If it falls within
        # the link gate of a frontier node, that link is added — and because that
        # frontier node may already have an outgoing edge from the primary pass, this
        # is exactly how true division events (2 outgoing edges) are produced.
        unmatched_new_idx = [c for c in range(len(new_ids)) if c not in matched_new]
        for c in unmatched_new_idx:
            dists_to_frontier = cost[:, c]
            nearest_r = int(np.argmin(dists_to_frontier))
            if dists_to_frontier[nearest_r] <= max_link_distance_um:
                edges.append((frontier_ids[nearest_r], new_ids[c]))
                matched_new.add(c)

        frontier = {new_ids[c]: nodes[new_ids[c]][1:] for c in range(len(new_ids))}
        gc.collect()

    return nodes, edges


# --------------------------------------------------------------------------------------
# 10. Competition metric — exact re-implementation
# --------------------------------------------------------------------------------------
def match_nodes_by_timepoint(pred_nodes, gt_nodes, scale, max_dist=METRIC_MAX_MATCH_DIST_UM):
    """Optimal per-timepoint bipartite matching of predicted to ground-truth nodes on
    scaled centroid distance, capped at max_dist micrometres."""
    pred_by_t = defaultdict(list)
    for nid, (t, z, y, x) in pred_nodes.items():
        pred_by_t[t].append((nid, z, y, x))
    gt_by_t = defaultdict(list)
    for nid, (t, z, y, x) in gt_nodes.items():
        gt_by_t[t].append((nid, z, y, x))

    pred_to_gt, gt_to_pred = {}, {}
    for t in set(pred_by_t) | set(gt_by_t):
        preds, gts = pred_by_t.get(t, []), gt_by_t.get(t, [])
        if not preds or not gts:
            continue
        pred_ids = [p[0] for p in preds]
        gt_ids = [g[0] for g in gts]
        pred_pos = np.array([p[1:] for p in preds], dtype=np.float64) * scale
        gt_pos = np.array([g[1:] for g in gts], dtype=np.float64) * scale

        cost = cdist(pred_pos, gt_pos)
        gated = np.where(cost <= max_dist, cost, 1e6)
        r_idx, c_idx = linear_sum_assignment(gated)
        for r, c in zip(r_idx, c_idx):
            if cost[r, c] <= max_dist:
                pred_to_gt[pred_ids[r]] = gt_ids[c]
                gt_to_pred[gt_ids[c]] = pred_ids[r]
    return pred_to_gt, gt_to_pred


def compute_edge_jaccard(pred_nodes, pred_edges, gt_nodes, gt_edges, scale,
                          estimated_total_nodes, max_dist=METRIC_MAX_MATCH_DIST_UM,
                          penalty_a=METRIC_NODE_PENALTY_A):
    """Adjusted Edge Jaccard for one sample, following the published metric exactly."""
    pred_to_gt, gt_to_pred = match_nodes_by_timepoint(pred_nodes, gt_nodes, scale, max_dist)

    gt_edge_set = set(gt_edges)
    gt_sources_of_target = defaultdict(set)
    gt_targets_of_source = defaultdict(set)
    for s, tt in gt_edges:
        gt_sources_of_target[tt].add(s)
        gt_targets_of_source[s].add(tt)

    TP, FP = 0, 0
    found_gt_edges = set()

    for (ps, pt) in pred_edges:
        gs = pred_to_gt.get(ps)
        gtt = pred_to_gt.get(pt)

        if gs is not None and gtt is not None and (gs, gtt) in gt_edge_set:
            TP += 1
            found_gt_edges.add((gs, gtt))
            continue

        is_fp = False
        if gtt is not None and gt_sources_of_target.get(gtt) and gs not in gt_sources_of_target[gtt]:
            is_fp = True
        if gs is not None and gt_targets_of_source.get(gs) and gtt not in gt_targets_of_source[gs]:
            is_fp = True
        if is_fp:
            FP += 1
        # else: predicted edge involves no annotated region -> ignored entirely

    FN = len(gt_edge_set) - len(found_gt_edges)
    denom = TP + FP + FN
    jaccard = TP / denom if denom > 0 else 0.0

    T_pred = len(pred_nodes)
    T_true = estimated_total_nodes if estimated_total_nodes and estimated_total_nodes > 0 else max(len(gt_nodes), 1)
    adjusted = jaccard * (1.0 - penalty_a * (T_pred - T_true) / T_true)
    adjusted = max(0.0, adjusted)

    return {"TP": TP, "FP": FP, "FN": FN, "jaccard": jaccard,
            "adjusted_jaccard": adjusted, "weight": denom,
            "pred_to_gt": pred_to_gt, "gt_to_pred": gt_to_pred}


def _gt_connected_components(gt_nodes, gt_edges):
    """Connected components of the GT annotation graph. Because annotation is sparse,
    this naturally fragments the true lineage forest into many small, separately
    'reliable' pieces — used to detect a predicted fork whose branches point to two
    genuinely different GT lineages (a merge/misassignment artifact)."""
    G = nx.Graph()
    G.add_nodes_from(gt_nodes.keys())
    G.add_edges_from(gt_edges)
    comp_of = {}
    for comp in nx.connected_components(G):
        fcomp = frozenset(comp)
        for n in comp:
            comp_of[n] = fcomp
    return comp_of


def _branch_is_unmerged(child, pred_children, pred_parents, fork):
    """Rule 5 (Unmerged branches): the direct child used by the fork must have that
    fork as its sole parent; if grandchild fallback is used, each grandchild must
    belong to that child alone. A locally merged/shared branch is rejected."""
    if pred_parents.get(child, []) != [fork]:
        return False
    for gk in pred_children.get(child, []):
        if pred_parents.get(gk, []) != [child]:
            return False
    return True


def _branch_evidence(child, pred_children, pred_to_gt, gt_component_of):
    """Rule 4 (Valid branch evidence): a matched direct child supplies its branch's
    evidence (and component) with precedence. If the child is unmatched, an
    unambiguous matched grandchild may supply it instead; if fallback grandchildren
    point to several GT components, the branch supplies no evidence at all."""
    grandkids = pred_children.get(child, [])
    if child in pred_to_gt:
        gt_id = pred_to_gt[child]
        return {gt_id}, gt_component_of.get(gt_id)
    matched_gks = [pred_to_gt[gk] for gk in grandkids if gk in pred_to_gt]
    comps = {gt_component_of.get(g) for g in matched_gks}
    comps.discard(None)
    if len(comps) == 1:
        return set(matched_gks), next(iter(comps))
    return set(), None  # no evidence, or ambiguous fallback -> no evidence


def _local_candidate_forks(D, gt_parents, gt_to_pred, pred_children, pred_fork_nodes):
    """Rule 1 (Local parent anchor) + the anchor-placement half of Rule 3 (Directed
    local topology): candidate forks for GT division D are predicted nodes matched to
    D or D's immediate predecessor (grandparent), or an immediate predicted successor
    of one of those matched anchors."""
    parent_side_gt = [D] + list(gt_parents.get(D, [])[:1])
    anchors = {gt_to_pred[n] for n in parent_side_gt if n in gt_to_pred}
    candidates = set()
    for anchor in anchors:
        if anchor in pred_fork_nodes:
            candidates.add(anchor)
        for succ in pred_children.get(anchor, []):
            if succ in pred_fork_nodes:
                candidates.add(succ)
    return candidates


def _evaluate_division_fork_pair(D, F, gt_children, pred_children, pred_parents,
                                  pred_to_gt, gt_component_of):
    """Rules 2-5 for a specific (GT division, candidate fork) pair: two distinct
    predicted branches (direct child of F, plus that child's immediate children) must
    each supply evidence for a different one of D's two daughter lineages (one child
    of D plus that child's immediate children), using only unmerged branches. Returns
    True/False; the actual branch pairing isn't needed beyond this go/no-go signal."""
    children_D = gt_children.get(D, [])[:2]
    if len(children_D) < 2:
        return False
    c1, c2 = children_D
    daughters = {
        0: {c1} | set(gt_children.get(c1, [])),
        1: {c2} | set(gt_children.get(c2, [])),
    }

    compat = {0: [], 1: []}
    for c in pred_children.get(F, []):
        if not _branch_is_unmerged(c, pred_children, pred_parents, F):
            continue  # Rule 5: merged/shared branches supply no evidence
        evidence_ids, _comp = _branch_evidence(c, pred_children, pred_to_gt, gt_component_of)
        for i in (0, 1):
            if evidence_ids & daughters[i]:
                compat[i].append(c)

    for c0 in compat[0]:
        for c1_ in compat[1]:
            if c0 != c1_:  # two DISTINCT branches, one per daughter lineage
                return True
    return False


def _fork_branch_conflict_and_merge(F, pred_children, pred_parents, pred_to_gt, gt_component_of):
    """Global per-fork check (independent of any specific GT division), used for FP
    conditions (c) and (d): do any two of this fork's branches carry evidence in
    different GT connected components, or is any of its branches locally merged?"""
    merged = False
    components = []
    for c in pred_children.get(F, []):
        if not _branch_is_unmerged(c, pred_children, pred_parents, F):
            merged = True
            continue
        _ids, comp = _branch_evidence(c, pred_children, pred_to_gt, gt_component_of)
        if comp is not None:
            components.append(comp)
    cross_component_conflict = len(set(components)) > 1
    return merged, cross_component_conflict


def compute_division_counts(pred_nodes, pred_edges, gt_nodes, gt_edges, scale,
                             pred_to_gt, gt_to_pred, max_dist=METRIC_MAX_MATCH_DIST_UM):
    """Exact re-implementation of the published Division Jaccard: a local window
    (grandparent -> dividing parent -> children -> grandchildren) around each GT
    division, directed local topology (no graph-wide reachability / weak-component
    shortcuts), unmerged-branch and cross-component evidence checks, and a genuine
    maximum-cardinality bipartite matching between candidate GT divisions and
    candidate predicted forks (each side capped at one partner)."""
    gt_children = defaultdict(list)
    gt_parents = defaultdict(list)
    gt_outdeg = defaultdict(int)
    for s, t in gt_edges:
        gt_children[s].append(t)
        gt_parents[t].append(s)
        gt_outdeg[s] += 1
    # "A ground-truth cell division has exactly two outgoing edges."
    gt_dividers = [n for n, d in gt_outdeg.items() if d == 2]

    pred_children = defaultdict(list)
    pred_parents = defaultdict(list)
    pred_outdeg = defaultdict(int)
    for s, t in pred_edges:
        pred_children[s].append(t)
        pred_parents[t].append(s)
        pred_outdeg[s] += 1
    # "Any predicted node with at least two outgoing edges is treated as a predicted fork."
    pred_fork_nodes = {n for n, d in pred_outdeg.items() if d >= 2}

    gt_component_of = _gt_connected_components(gt_nodes, gt_edges)

    # ---- Candidate (D, F) pairs: local anchor test, then the full rule set ----
    local_candidates_by_division = {}
    all_local_candidate_forks = set()
    B = nx.Graph()
    gt_side_nodes = []
    for D in gt_dividers:
        node_D = ("D", D)
        B.add_node(node_D, bipartite=0)
        gt_side_nodes.append(node_D)

        candidates = _local_candidate_forks(D, gt_parents, gt_to_pred, pred_children, pred_fork_nodes)
        local_candidates_by_division[D] = candidates
        all_local_candidate_forks |= candidates

        for F in candidates:
            if _evaluate_division_fork_pair(D, F, gt_children, pred_children, pred_parents,
                                             pred_to_gt, gt_component_of):
                node_F = ("F", F)
                B.add_node(node_F, bipartite=1)
                B.add_edge(node_D, node_F)

    # ---- Maximum-cardinality bipartite matching: each side capped at one partner ----
    if gt_side_nodes:
        matching = nx.algorithms.bipartite.maximum_matching(B, top_nodes=gt_side_nodes)
    else:
        matching = {}
    tp_pairs = {node_D[1]: matching[node_D][1] for node_D in gt_side_nodes if node_D in matching}

    TP = len(tp_pairs)
    FN = len(gt_dividers) - TP
    tp_paired_forks = set(tp_pairs.values())

    # ---- False positives: union of forks flagged by any of the four evidence rules ----
    fp_forks = set()
    for f in pred_fork_nodes:
        matched_gt = pred_to_gt.get(f)
        if matched_gt is not None and gt_outdeg.get(matched_gt, 0) >= 1:
            fp_forks.add(f)  # (a) fork matches an annotated (non-leaf) GT node
        merged, cross_conflict = _fork_branch_conflict_and_merge(
            f, pred_children, pred_parents, pred_to_gt, gt_component_of)
        if cross_conflict:
            fp_forks.add(f)  # (c) branches point to different GT components
        if merged:
            fp_forks.add(f)  # (d) locally merged/shared branch
    for f in all_local_candidate_forks:
        if f not in tp_paired_forks:
            fp_forks.add(f)  # (b) local candidate that failed topology or lost the pairing

    fp_forks -= tp_paired_forks  # a TP fork is never also an FP
    FP = len(fp_forks)

    return TP, FP, FN


def evaluate_dataset(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated_total_nodes, scale):
    edge_stats = compute_edge_jaccard(pred_nodes, pred_edges, gt_nodes, gt_edges,
                                       scale, estimated_total_nodes)
    div_tp, div_fp, div_fn = compute_division_counts(
        pred_nodes, pred_edges, gt_nodes, gt_edges, scale,
        edge_stats["pred_to_gt"], edge_stats["gt_to_pred"])
    return edge_stats, (div_tp, div_fp, div_fn)


def combined_score(per_sample_edge_stats, per_sample_division_counts):
    total_w = sum(s["weight"] for s in per_sample_edge_stats)
    if total_w > 0:
        adjusted_edge_jaccard = sum(s["adjusted_jaccard"] * s["weight"]
                                     for s in per_sample_edge_stats) / total_w
    else:
        adjusted_edge_jaccard = 0.0

    tp_sum = sum(c[0] for c in per_sample_division_counts)
    fp_sum = sum(c[1] for c in per_sample_division_counts)
    fn_sum = sum(c[2] for c in per_sample_division_counts)
    denom = tp_sum + fp_sum + fn_sum
    division_jaccard = tp_sum / denom if denom > 0 else 0.0

    score = adjusted_edge_jaccard + METRIC_DIVISION_WEIGHT * division_jaccard
    return {
        "adjusted_edge_jaccard": adjusted_edge_jaccard,
        "division_jaccard": division_jaccard,
        "division_TP": tp_sum, "division_FP": fp_sum, "division_FN": fn_sum,
        "final_score": score,
    }


def run_tracking_validation(dataset_ids, datasets_index, scale, cnn_model, rf_model,
                             device, ensemble_threshold, desc="validating",
                             max_datasets=None, max_timepoints=None, t_start=None, reserve=0.0):
    """Local competition-metric proxy score. `max_datasets`/`max_timepoints` bound its
    runtime (this is a fast proxy for tuning, not the exact full-video score Kaggle will
    compute on the actual submission). `t_start`/`reserve` add a hard wall-clock cutoff."""
    if max_datasets is not None and len(dataset_ids) > max_datasets:
        dataset_ids = random.sample(list(dataset_ids), max_datasets)

    per_sample_edge_stats, per_sample_division_counts = [], []
    for dataset_id in tqdm(dataset_ids, desc=desc):
        if t_start is not None and deadline_hit(t_start, reserve):
            print(f"[{desc}] runtime budget reached, stopping early")
            break
        entry = datasets_index["train"].get(dataset_id)
        if entry is None or entry["geff"] is None:
            continue
        try:
            img_arr = open_image_array(entry["zarr"])
            pred_nodes, pred_edges = track_dataset(
                img_arr, scale, cnn_model=cnn_model, rf_model=rf_model, device=device,
                ensemble_threshold=ensemble_threshold, show_progress=False,
                max_timepoints=max_timepoints)
            gt_nodes, gt_edges, estimated_total = load_geff_graph(entry["geff"])
            edge_stats, div_counts = evaluate_dataset(
                pred_nodes, pred_edges, gt_nodes, gt_edges, estimated_total, scale)
            per_sample_edge_stats.append(edge_stats)
            per_sample_division_counts.append(div_counts)
        except Exception as exc:
            print(f"[{desc}] skipped {dataset_id}: {exc}")
        finally:
            gc.collect()
    return combined_score(per_sample_edge_stats, per_sample_division_counts) if per_sample_edge_stats else None


# --------------------------------------------------------------------------------------
# 11. Submission writer
# --------------------------------------------------------------------------------------
def build_submission_rows(dataset_id, nodes, edges):
    rows = []
    for nid, (t, z, y, x) in nodes.items():
        rows.append({"dataset": dataset_id, "row_type": "node", "node_id": nid,
                      "t": t, "z": z, "y": y, "x": x, "source_id": -1, "target_id": -1})
    for src, tgt in edges:
        rows.append({"dataset": dataset_id, "row_type": "edge", "node_id": -1,
                      "t": -1, "z": -1, "y": -1, "x": -1, "source_id": src, "target_id": tgt})
    return rows


def write_submission(all_rows, out_path=OUTPUT_CSV):
    df = pd.DataFrame(all_rows, columns=["dataset", "row_type", "node_id", "t", "z", "y",
                                          "x", "source_id", "target_id"])
    df.insert(0, "id", np.arange(len(df)))
    df.to_csv(out_path, index=False)
    return df


# --------------------------------------------------------------------------------------
# 12. Main execution
# --------------------------------------------------------------------------------------
if __name__ == "__main__":
    t_start = time.time()
    os.makedirs(PLOTS_DIR, exist_ok=True)
    log_memory("start")
    print(f"Runtime budget: {TOTAL_RUNTIME_BUDGET_SECONDS/3600:.1f}h total, "
          f"with {TEST_STAGE_RESERVED_SECONDS/60:.0f}min always reserved for the test-set submission.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    datasets = discover_datasets(ROOT_PATH)
    print(f"Discovered {len(datasets['train'])} train, {len(datasets['test'])} test, "
          f"{len(datasets['unknown'])} unclassified samples under {ROOT_PATH}")

    train_dataset_ids = [d for d, e in datasets["train"].items() if e["geff"] is not None]
    if MAX_TRAIN_DATASETS_FOR_VALIDATION is not None:
        train_dataset_ids = train_dataset_ids[:MAX_TRAIN_DATASETS_FOR_VALIDATION]
    # The previous run built classifier training data from all 199 annotated datasets and
    # then fully tracked every train + holdout dataset for local validation — that's what
    # drove the >10 hour runtime. Subsample the pool up front instead: everything derived
    # from this smaller pool (the four-way split, candidate-building, validation) shrinks
    # proportionally, and the final test-set submission below is unaffected by this cap.
    if len(train_dataset_ids) > MAX_DATASETS_FOR_PIPELINE:
        print(f"Subsampling {MAX_DATASETS_FOR_PIPELINE} of {len(train_dataset_ids)} annotated "
              f"training datasets to keep total runtime within budget.")
        train_dataset_ids = random.sample(train_dataset_ids, MAX_DATASETS_FOR_PIPELINE)

    cnn_model, rf_model, ensemble_threshold = None, None, 0.5
    split_train_ids = split_val_ids = split_test_ids = split_holdout_ids = None
    Xp_train = Xf_train = Xp_val = Xf_val = Xp_test = Xf_test = Xp_holdout = Xf_holdout = None

    # ================= Candidate classifier training =================
    if len(train_dataset_ids) >= 4:
        # ---- 1. Four-way split at the dataset (embryo/sample) level, no leakage ----
        split_train_ids, split_val_ids, split_test_ids, split_holdout_ids = split_datasets_four_way(train_dataset_ids)
        check_domain_relevance(split_train_ids, split_holdout_ids)
        check_domain_relevance(split_train_ids, split_val_ids)
        check_domain_relevance(split_train_ids, split_test_ids)
        print(f"Split sizes -> train:{len(split_train_ids)} val:{len(split_val_ids)} "
              f"test:{len(split_test_ids)} holdout:{len(split_holdout_ids)}")

        # ---- 2. Build labelled candidate-region dataset per split (budget-aware) ----
        build_reserve = TEST_STAGE_RESERVED_SECONDS + 600  # also keep time for validation + training
        Xp_train, Xf_train, y_train, _g_train = build_candidate_label_dataset(
            split_train_ids, datasets, PHYSICAL_SCALE, t_start=t_start, reserve=build_reserve)
        Xp_val, Xf_val, y_val, _g_val = build_candidate_label_dataset(
            split_val_ids, datasets, PHYSICAL_SCALE, t_start=t_start, reserve=build_reserve)
        Xp_test, Xf_test, y_test, _g_test = build_candidate_label_dataset(
            split_test_ids, datasets, PHYSICAL_SCALE, t_start=t_start, reserve=build_reserve)
        Xp_holdout, Xf_holdout, y_holdout, _g_holdout = build_candidate_label_dataset(
            split_holdout_ids, datasets, PHYSICAL_SCALE, t_start=t_start, reserve=build_reserve)

        verify_data_splits(y_train, y_val, y_test, y_holdout)
        log_memory("after candidate-dataset build")
        print(f"[timing] candidate-dataset build done at {time.time()-t_start:.0f}s")

        if len(y_train) >= 20 and len(y_val) >= 5 and len(np.unique(y_train)) > 1:
            # ---- 3. Outlier detection & sample-weight adjustment (train split only) ----
            w_train = detect_outliers_and_weights(Xf_train)

            # ---- 4. Hyperparameter tuning ----
            best_cfg = randomized_hyperparameter_search(Xp_train, y_train, w_train, Xp_val, y_val, device)

            # ---- 5. Final training with early stopping + best hyperparameters ----
            cnn_model, history, best_val_acc = train_cnn_model(
                Xp_train, y_train, w_train, Xp_val, y_val, device=device, **best_cfg)

            # ---- 6. Classical model (for the ensemble) + cross-validation ----
            rf_model = train_classical_model(Xf_train, y_train, w_train)
            if len(y_train) + len(y_val) >= 30:
                cross_validate_classical_model(np.concatenate([Xf_train, Xf_val]),
                                                np.concatenate([y_train, y_val]))

            # ---- 7. Model selection: CNN-only vs RF-only vs ensemble, by val accuracy ----
            _, cnn_val_acc = evaluate_cnn(cnn_model, Xp_val, y_val, device)
            rf_val_probs = rf_model.predict_proba(Xf_val)[:, 1] if len(y_val) else np.zeros(0)
            rf_val_acc = float(np.mean((rf_val_probs >= 0.5) == y_val)) if len(y_val) else float("nan")
            _, _, ens_val_acc = evaluate_ensemble(cnn_model, rf_model, Xp_val, Xf_val, y_val, device)
            print(f"[model selection] cnn_val_acc={cnn_val_acc:.4f} rf_val_acc={rf_val_acc:.4f} "
                  f"ensemble_val_acc={ens_val_acc:.4f}")
            candidates_by_acc = [("cnn", cnn_val_acc), ("rf", rf_val_acc), ("ensemble", ens_val_acc)]
            candidates_by_acc = [(name, acc) for name, acc in candidates_by_acc if not np.isnan(acc)]
            model_choice = max(candidates_by_acc, key=lambda kv: kv[1])[0] if candidates_by_acc else "ensemble"
            print(f"[model selection] best model: {model_choice}")
            if model_choice == "cnn":
                rf_model_for_pipeline = None
            elif model_choice == "rf":
                cnn_model, rf_model_for_pipeline = None, rf_model
            else:
                rf_model_for_pipeline = rf_model
            rf_model = rf_model_for_pipeline

            # ---- 8. Full evaluation across all splits (train/val/test/holdout) ----
            _, train_acc = evaluate_cnn(cnn_model, Xp_train, y_train, device) if cnn_model is not None else (None, float("nan"))
            test_probs, _test_preds, test_acc = evaluate_ensemble(cnn_model, rf_model, Xp_test, Xf_test, y_test, device)
            _, _, holdout_acc = evaluate_ensemble(cnn_model, rf_model, Xp_holdout, Xf_holdout, y_holdout, device)

            split_reports = {"train": train_acc, "val": best_val_acc, "test": test_acc, "holdout": holdout_acc}
            print(f"[final evaluation across splits, incl. unseen holdout] {split_reports}")

            if cnn_model is not None:
                plot_training_report(history, split_reports, y_test, test_probs)

            # ---- 9. Save the best model(s) ----
            if cnn_model is not None:
                torch.save(cnn_model.state_dict(), "best_cnn_model.pt")
                print("Saved best_cnn_model.pt")
            if rf_model is not None:
                joblib.dump(rf_model, "best_rf_model.joblib")
                print("Saved best_rf_model.joblib")
        else:
            print("Not enough labelled candidate data to train the classifier reliably; "
                  "tracking will fall back to plain area-threshold detection.")
            cnn_model, rf_model = None, None
    else:
        print("Fewer than 4 annotated training datasets found; skipping classifier training "
              "(need >=1 dataset per split for a leakage-free four-way split).")

    del Xp_train, Xf_train, Xp_val, Xf_val, Xp_test, Xf_test, Xp_holdout, Xf_holdout
    gc.collect()
    log_memory("after classifier training")
    print(f"[timing] classifier training done at {time.time()-t_start:.0f}s")

    # ================= Competition-metric validation (tracking pipeline) =================
    # A fast, bounded PROXY of the real score (capped datasets + capped frames), purely to
    # gauge quality before committing the remaining budget to the actual test submission.
    validation_reserve = TEST_STAGE_RESERVED_SECONDS
    if split_train_ids is not None:
        train_report = run_tracking_validation(
            split_train_ids, datasets, PHYSICAL_SCALE, cnn_model, rf_model, device,
            ensemble_threshold, desc="tracking-metric on train split (proxy)",
            max_datasets=TRACKING_VALIDATION_MAX_DATASETS_PER_SPLIT,
            max_timepoints=MAX_TIMEPOINTS_FOR_TRACKING_VALIDATION,
            t_start=t_start, reserve=validation_reserve)
        holdout_report = run_tracking_validation(
            split_holdout_ids, datasets, PHYSICAL_SCALE, cnn_model, rf_model, device,
            ensemble_threshold, desc="tracking-metric on HELD-OUT split (proxy, unseen)",
            max_datasets=TRACKING_VALIDATION_MAX_DATASETS_PER_SPLIT,
            max_timepoints=MAX_TIMEPOINTS_FOR_TRACKING_VALIDATION,
            t_start=t_start, reserve=validation_reserve)
    else:
        train_report = run_tracking_validation(
            train_dataset_ids, datasets, PHYSICAL_SCALE, cnn_model, rf_model, device,
            ensemble_threshold, desc="tracking-metric on train (proxy)",
            max_datasets=TRACKING_VALIDATION_MAX_DATASETS_PER_SPLIT,
            max_timepoints=MAX_TIMEPOINTS_FOR_TRACKING_VALIDATION,
            t_start=t_start, reserve=validation_reserve)
        holdout_report = None

    print("\n=== Competition-metric validation (proxy, capped datasets/frames): TRAIN split ===")
    print(train_report)
    if holdout_report is not None:
        print("\n=== Competition-metric validation (proxy, capped datasets/frames): HOLDOUT (unseen) ===")
        print(holdout_report)

    log_memory("after validation")
    print(f"[timing] validation done at {time.time()-t_start:.0f}s")

    # ================= Build predictions for the test split (mandatory, full-length) =====
    # This stage always tracks every frame of every test dataset — no caps — since it's
    # what actually gets graded. If the wall-clock budget is nearly spent by the time we
    # get here, remaining datasets fall back to a cheap single-node placeholder so every
    # required dataset id still appears in submission.csv rather than being skipped.
    test_ids = list(datasets["test"].keys())
    all_rows = []
    for dataset_id in tqdm(test_ids, desc="predicting on test"):
        entry = datasets["test"][dataset_id]
        out_of_time = deadline_hit(t_start, reserve=0.0)
        try:
            img_arr = open_image_array(entry["zarr"])
            if out_of_time:
                print(f"[test] runtime budget exhausted — using placeholder for {dataset_id}")
                t0_vol = np.asarray(img_arr[0])
                cz, cy, cx = tuple(s // 2 for s in t0_vol.shape)
                nodes, edges = {1: (0, cz, cy, cx)}, []
                del t0_vol
            else:
                nodes, edges = track_dataset(img_arr, PHYSICAL_SCALE, cnn_model=cnn_model, rf_model=rf_model,
                                              device=device, ensemble_threshold=ensemble_threshold,
                                              show_progress=False)
                if not nodes:
                    t0_vol = np.asarray(img_arr[0])
                    cz, cy, cx = tuple(s // 2 for s in t0_vol.shape)
                    nodes = {1: (0, cz, cy, cx)}
                    edges = []
                    del t0_vol
        except Exception as exc:
            print(f"[test] failed on {dataset_id}: {exc}")
            nodes, edges = {1: (0, 0, 0, 0)}, []
        all_rows.extend(build_submission_rows(dataset_id, nodes, edges))
        gc.collect()

    submission_df = write_submission(all_rows, OUTPUT_CSV)
    print(f"\nWrote {OUTPUT_CSV} with {len(submission_df)} rows across {len(test_ids)} datasets.")
    log_memory("end")
    print(f"Total runtime: {time.time() - t_start:.1f}s ({(time.time() - t_start)/3600:.2f}h)")